In [3]:
from pathlib import Path

ROOT = Path.cwd()

DATA_DIR = ROOT / "parquet"

In [4]:
import pandas as pd

amazon_books = pd.read_parquet(DATA_DIR / "amazon.parquet")
goodreads = pd.read_parquet(DATA_DIR / "goodreads.parquet")
metabooks = pd.read_parquet(DATA_DIR / "metabooks.parquet")

In [5]:
def add_row_id(df: pd.DataFrame, prefix: str, start: int = 1, colname: str | None = None):
    """
    Adds a stable row identifier like 'amazon_1', 'amazon_2', ...
    Inserts it as the FIRST column and returns the modified DataFrame.

    Args:
        df:       Input DataFrame.
        prefix:   String prefix for IDs (e.g., "amazon", "metabooks", "goodreads").
        start:    Starting index (default 1).
        colname:  Name of the ID column; defaults to f"{prefix}_row_id".
    """
    if colname is None:
        colname = f"{prefix}"
    seq = range(start, start + len(df))
    ids = [f"{prefix}_{i}" for i in seq]
    out = df.copy()
    out.insert(0, colname, ids) 
    return out

In [6]:
amazon_books = add_row_id(amazon_books, "amazon", colname="id")
goodreads = add_row_id(goodreads, "goodreads", colname="id")
metabooks = add_row_id(metabooks, "metabooks", colname="id")

In [7]:
import re

def clean_text(t):
  t = str(t).lower()
  t = re.sub(r'<.*?>', '', t)
  t = re.sub(r'[^a-z0-9\s]', '', t)
  t = re.sub(r'\s+', ' ', t).strip()
  return t

amazon_books['title_norm'] = amazon_books['title'].apply(clean_text)
amazon_books['author_norm'] = amazon_books['Book-Author'].apply(clean_text)

goodreads['title_norm'] = goodreads['title'].apply(clean_text)
goodreads['author_norm'] = goodreads['author'].apply(clean_text)

metabooks['title_norm'] = metabooks['title'].apply(clean_text)
metabooks['author_norm'] = metabooks['Author_Name'].apply(clean_text)

In [13]:
amazon_books = amazon_books.rename(columns={"Year-Of-Publication": "publish_year",
                                            "ISBN_clean": "isbn_clean",
                                            "Book-Author": "author",
                                            "Publisher": "publisher"})

goodreads = goodreads.rename(columns={"numRatings": "num_ratings",
                                      "Publish_Year": "publish_year",
                                      "ISBN_clean": "isbn_clean",
                                      "bookFormat": "book_format",
                                      "pages": "page_count",
                                      "num_ratings":"rating_number"})

metabooks = metabooks.rename(columns={"Author_Name": "author",
                                      "Language": "language",
                                      "Publisher": "publisher",
                                      "Page_Count": "page_count",
                                      "Publish_Year": "publish_year",
                                      "ISBN_clean": "isbn_clean",
                                      "average_rating": "rating",
                                      "categories": "genres"})

In [14]:
amazon_books.to_parquet(DATA_DIR / "amazon_new.parquet", index=False, compression="snappy")
goodreads.to_parquet(DATA_DIR / "goodreads_new.parquet", index=False, compression="snappy")
metabooks.to_parquet(DATA_DIR / "metabooks_new.parquet", index=False, compression="snappy")